In [ ]:
from functools import partial
from pathlib import Path
import os
import shutil

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ─────────────────────────────────────────────────────────────────────────────

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# Get the number of usable cores for parallel processing by looking at the system (n-2)
num_usable_cores = os.cpu_count() - 2
print(f"Number of usable cores for parallel processing: {num_usable_cores}")

# update weather flag
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"

# new_weather_epw = "Lecco_IT_TMY"
# new_weather_climate_zone = "4A"

### Activity 0: Did everything install correctly?

In [ ]:
baseline_analysis_dir = analysis_dir / "activity_0"
if not baseline_analysis_dir.exists():
    baseline_analysis_dir.mkdir()

# Just print out one to make sure it looks right.
uo_test = UO(baseline_analysis_dir, "test", template_dir=template_data_dir)
uo_test.info()

### Activity 1: Example projects

In [ ]:
activity_1_analysis_dir = analysis_dir / "activity_1"
if not activity_1_analysis_dir.exists():
    activity_1_analysis_dir.mkdir()

uo_coincident = UO(
    activity_1_analysis_dir, "coincident_pre", template_dir=template_data_dir
)
uo_diverse = UO(activity_1_analysis_dir, "diverse_pre", template_dir=template_data_dir)

uo_coincident.create_example_coincident_project()
uo_diverse.create_example_diverse_project()

uo_coincident.create_scenarios("class_project_coincident.json")
uo_diverse.create_scenarios("class_project_diverse.json")

# copy over to the coincident and diverse directories for activity 1
uo_coincident = uo_coincident.update_project_files("coincident")
uo_diverse = uo_diverse.update_project_files("diverse")

# run sims faster by adding more parallel processing
uo_coincident.set_number_parallel(num_usable_cores)
uo_diverse.set_number_parallel(num_usable_cores)

# copy over the weather files
uo_coincident.copy_over_weather()
uo_diverse.copy_over_weather()

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )
    uo_diverse.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )

#### Fix some items in the feature file due to dependency changes in the underlying libraries.

In [ ]:
# Copy over the updated Baseline.rb mapper file
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_coincident.project_path / "mappers" / "Baseline.rb",
)
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_diverse.project_path / "mappers" / "Baseline.rb",
)

In [ ]:
# Run the diverse baseline scenario
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

In [ ]:
# Run the diverse baseline scenario
uo_diverse.run("class_project_diverse.json", "baseline_scenario.csv")

In [ ]:
# post process the scenarios for both projects
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_diverse.process_scenario("class_project_diverse.json", "baseline_scenario.csv")

In [ ]:
# visualize both the projects
uo_coincident.visualize_feature("class_project_coincident.json")
uo_diverse.visualize_feature("class_project_diverse.json")

# and then the feature visualization to compare across the project.
# There is only one scenario in this activity, so it doesn't show much.
uo_coincident.visualize_feature("class_project_coincident.json")

### Activity 2 EEMs


In [ ]:
# This is the same as above, but in a new directory
activity_2_analysis_dir = analysis_dir / "activity_2"
if not activity_2_analysis_dir.exists():
    activity_2_analysis_dir.mkdir()

uo_coincident = UO(
    activity_2_analysis_dir, "coincident_pre", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

uo_coincident.create_scenarios("class_project_coincident.json")

# copy over to the coincident and diverse directories for activity 2
uo_coincident = uo_coincident.update_project_files("coincident")

# run sims faster
uo_coincident.set_number_parallel(num_usable_cores)

# copy over the weather files
uo_coincident.copy_over_weather()

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )

# Fix issues -- copy over the updated Baseline.rb mapper file
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_coincident.project_path / "mappers" / "Baseline.rb",
)
# Copy over the base workflow from the template. This one includes all the EEMs that are needed in the class project since
# the class_project_workflow.json inherits from base_workflow.osw. This is a workaround to avoid having to patch the workflow file to add in the missing EEM steps.
shutil.copy2(
    template_data_dir / "mappers" / "base_workflow.osw",
    uo_coincident.project_path / "mappers" / "base_workflow.osw",
)

#### Enable some of the measures in the ClassProject.rb file

In [ ]:
def enable_classproject_measures(uo_wrapper):
    """
    Flip __SKIP__ from true -> false for the four requested EEM measures.
    """
    mapper_path = uo_wrapper.project_path / "mappers" / "ClassProject.rb"
    text = mapper_path.read_text(encoding="utf-8")

    measures_to_enable = [
        "ReduceElectricEquipmentLoadsByPercentage",
        "ReduceSpaceInfiltrationByPercentage",
        "EnableEconomizerControl",
        "AddOverhangsByProjectionFactor",
    ]

    changed_measures = []
    for measure in measures_to_enable:
        old = f"OpenStudio::Extension.set_measure_argument(osw, '{measure}', '__SKIP__', true)"
        new = f"OpenStudio::Extension.set_measure_argument(osw, '{measure}', '__SKIP__', false)"
        if old in text:
            text = text.replace(old, new)
            changed_measures.append(measure)

    mapper_path.write_text(text, encoding="utf-8")
    print(f"Enabled measures: {changed_measures}")


enable_classproject_measures(uo_coincident)

In [ ]:
# Rerun the baseline, because this is a new directory (old data are in another folder)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

In [ ]:
# The need to create a new mapper (per the instructions) is not needed, since there is
# a classproject_scenario.csv, just run that one.

# run the class project with the updated measures that were selected
uo_coincident.run("class_project_coincident.json", "classproject_scenario.csv")

In [ ]:
# process the scenarios
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.process_scenario(
    "class_project_coincident.json", "classproject_scenario.csv"
)

# create the scenario and feature visualizations
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "classproject_scenario.csv"
)

# and then the feature visualization to compare the scenarios in the project.
uo_coincident.visualize_feature("class_project_coincident.json")